# PDF Parsing Agent with Gemma 3 + Nemotron VLM on Google ADK

This notebook builds an **agentic PDF-parsing pipeline** that combines two models orchestrated by the [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/):

| Component | Role |
|---|---|
| **Gemma 3 12B** (Vertex AI) | Orchestrator — decides *when* to call a tool and summarises the results |
| **Nemotron Nano VLM** (Vertex AI) | Tool — extracts text from PDF page images via vision-language understanding |
| **Google ADK Runner** | Manages the multi-turn conversation, dispatches tool calls, and feeds results back |

### How it works

1. The user asks a question about a PDF document.
2. Gemma generates a native `tool_code` block (following the [Gemma function-calling cookbook](https://github.com/google-gemini/gemma-cookbook)) calling `parse_pdf_tool`.
3. The ADK Runner intercepts the tool call, executes it — which sends every PDF page to Nemotron VLM for OCR/extraction.
4. The extracted text is returned to Gemma as a `tool_output` block.
5. Gemma produces a final answer grounded in the real document content.

## 0. Setup

### Install dependencies

In [ ]:
%pip install --upgrade pip -q
%pip install -r requirements.txt --upgrade -q

In [ ]:
### ⚠️ The next cell will Restart the Runtime (Important)

### To use the newly installed packages (like `google-adk`), we must restart the active Python kernel.

### **What to expect:**
### 1. Running the cell below will trigger a system restart.
### 2. You will likely see a popup message saying **"The kernel appears to have died. It will restart automatically."**
### 3. **This is normal and expected.** You can safely ignore the message or click "OK".
### 4. Wait about 5-10 seconds for the kernel to come back online before running the next cell.

In [ ]:
import IPython

print("Restarting the kernel to apply package updates...")

app = IPython.Application.instance()
# This command triggers the restart, causing the "Kernel Died" popup
app.kernel.do_shutdown(True)

Restarting the kernel after installs with the command above, allows the environment to access the new packages

### Authenticate with Google Cloud

Run the following command **in your terminal** {Go to Top left "File"->"New"->"Terminal" to set up Application Default Credentials (ADC) by running gcloud auth application-default login. This is required for accessing Vertex AI endpoints.


```bash
gcloud auth application-default login
```

## 1. Imports

In [ ]:
import warnings
# Suppress warnings for a clean demo experience
warnings.filterwarnings("ignore", category=FutureWarning)

import ast
import asyncio
import base64
import json
import re
import os
from io import BytesIO
from pathlib import Path
from typing import Any, AsyncGenerator

# Third-party imports
import fitz  # PyMuPDF
import vertexai
import google.auth
import google.auth.transport.requests
from google.cloud import aiplatform
from openai import OpenAI
from PIL import Image

# Google ADK imports
from google.adk import Runner
from google.adk.agents import Agent
from google.adk.models.base_llm import BaseLlm
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Patch asyncio to allow nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

print("Imports complete.")

## 2. Configuration

Point to your Vertex AI project, region, and endpoint display-names. `PDF_PATH` is pre-set.

In [ ]:
import google.auth
from google.cloud import aiplatform

# 1. Auto-detect and print the Project ID
_, PROJECT = google.auth.default()
print(f"Current Project = {PROJECT}")

# 2. Initialize Vertex AI to list endpoints
aiplatform.init(project=PROJECT, location="us-central1")

# 3. List all active endpoints for easy copy-pasting
print("\nAvailable Endpoints (Copy the 2 Relevant Endpoint Names (for Gemma & Nemotron) & Location for the next cell):")
endpoints = aiplatform.Endpoint.list()
if not endpoints:
    print("   (No endpoints found in this project/region)")
else:
    for ep in endpoints:
        print(f"\n🔹 Endpoint Name = {ep.display_name}")
        print(f"Location= {ep.location}")
        #print(f"   • Resource Name=  {ep.resource_name}")
        #print(f"   • Endpoint ID= {ep.name}")
       
    

In [ ]:
LOCATION = "us-central1" #LOCATION based on output from above cell (both models must be in the same location)


GEMMA_ENDPOINT_NAME = "gemma-3-12b-it-mg-one-click-deploy" #UPDATE GEMMA_ENDPOINT_NAME based on output from above cell
VLM_ENDPOINT_NAME = "nvidia_nemotron-nano-12b-v2-vl_1_5_0-mg-one-click-deploy" #UPDATE VLM_ENDPOINT_NAME based on output from above cell



VLM_MODEL_ID = "nvidia/nemotron-nano-12b-v2-vl" #Don't change
GEMMA_MODEL_ID ="gemma-3-12b-it" #Don't change

PDF_PATH = Path("v2-NVDA-F3Q26-Quarterly-Presentation.pdf") #Don't change

BATCH_SIZE = 1
EXTRACT_PROMPT = (
    "Extract all visible text from these slides. "
    "1. STRUCTURE: Transcribe the SLIDE TITLE first. "
    "2. CHARTS & LABELS: If a text label appears near a numerical value (e.g., in a bar chart), "
    "   transcribe them together as 'Label: Value' (e.g., 'Revenue: $57.0B'). "
    "3. TABLES: Use Markdown format with '|' separators. "
    "4. Do not summarize. Transcription must be exact."
)

vertexai.init(project=PROJECT, location=LOCATION)
aiplatform.init(project=PROJECT, location=LOCATION)
print("Vertex AI initialized.")

## 3. Vertex AI Helpers

Two small utilities:
- `get_endpoint` — looks up a Vertex AI endpoint by display-name.
- `get_gemma_client` — returns an OpenAI-compatible client pointed at the Gemma endpoint (uses your ADC credentials).

In [4]:
def get_endpoint(name: str) -> aiplatform.Endpoint:
    """Find a Vertex AI endpoint whose display name contains *name*."""
    endpoints = aiplatform.Endpoint.list()
    target = next((ep for ep in endpoints if name in ep.display_name), None)
    if not target:
        raise RuntimeError(f"Endpoint containing '{name}' not found")
    return target


def get_gemma_client(endpoint: aiplatform.Endpoint) -> OpenAI:
    """Create an OpenAI-compatible client for a Vertex AI Gemma endpoint."""
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    base_url = (
        f"https://{LOCATION}-aiplatform.googleapis.com"
        f"/v1beta1/{endpoint.resource_name}"
    )
    if (
        hasattr(endpoint.gca_resource, "dedicated_endpoint_dns")
        and endpoint.gca_resource.dedicated_endpoint_dns
    ):
        base_url = (
            f"https://{endpoint.gca_resource.dedicated_endpoint_dns}"
            f"/v1beta1/{endpoint.resource_name}"
        )
    return OpenAI(base_url=base_url, api_key=creds.token)

## 4. PDF / VLM Utilities

These functions handle the heavy lifting of converting a PDF into text:

1. **`load_pdf_pages`** — renders every page of the PDF to a PIL image using PyMuPDF.
2. **`encode_pil_to_data_url`** — converts a PIL image to a base64 data-URL for the VLM.
3. **`vlm_extract`** — sends a batch of images to the Nemotron VLM endpoint via `raw_predict`.
4. **`extract_pdf`** — orchestrates the above in batches of `BATCH_SIZE` pages.

In [ ]:
def encode_pil_to_data_url(pil_image: Image.Image) -> str:
    """Convert a PIL image to a JPEG base64 data-URL."""
    if pil_image.mode not in ("RGB",):
        if pil_image.mode in ("RGBA", "LA") or (
            pil_image.mode == "P" and "transparency" in pil_image.info
        ):
            bg = Image.new("RGB", pil_image.size, (255, 255, 255))
            bg.paste(pil_image.convert("RGBA"), mask=pil_image.convert("RGBA").split()[-1])
            pil_image = bg
        else:
            pil_image = pil_image.convert("RGB")
    buf = BytesIO()
    pil_image.save(buf, format="JPEG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/jpeg;base64,{b64}"

from IPython.display import display

def load_pdf_pages(pdf_path: Path) -> list[Image.Image]:
    """Render every page of a PDF to a PIL image."""
    pages = []
    with fitz.open(pdf_path) as pdf:
        for page in pdf:
            pix = page.get_pixmap(dpi=150)
            mode = "RGBA" if pix.alpha else "RGB"
            pages.append(Image.frombytes(mode, [pix.width, pix.height], pix.samples))
    return pages

def preview_pdf(pdf_path: Path, page_numbers: list[int] = None):
    """
    Visual helper to show specific pages of the document.
    
    Args:
        pdf_path (Path): Path to the PDF file.
        page_numbers (list[int]): A list of page numbers to display (1-based index). 
                                  If None, displays the first page.
    """
    # Load all pages first
    pages = load_pdf_pages(pdf_path)
    total_pages = len(pages)
    
    if not pages:
        print("⚠️ The PDF is empty or could not be read.")
        return

    # Default to Page 1 if no specific pages requested
    if page_numbers is None:
        page_numbers = [1]
        
    print(f"📄 Loaded {total_pages} pages from '{pdf_path.name}'.")
    
    for p_num in page_numbers:
        # Check for valid page range (1 to total_pages)
        if 1 <= p_num <= total_pages:
            print(f"\n--- Previewing Page {p_num} ---")
            # Convert 1-based index to 0-based index
            img = pages[p_num - 1].copy()
            
            # Resize for display (keeps the notebook tidy)
            img.thumbnail((500, 500)) 
            display(img)
        else:
            print(f"⚠️ Page {p_num} is out of range (1-{total_pages}). Skipping.")

preview_pdf(PDF_PATH, page_numbers=[1, 3, 9, 10])
        
def vlm_extract(target: aiplatform.Endpoint, image_urls: list[str]) -> str:
    """Send a batch of page images to the Nemotron VLM and return extracted text."""
    body = json.dumps({
        "model": VLM_MODEL_ID,
        "messages": [
            {"role": "system", "content": "/no_think"},
            {
                "role": "user",
                "content": [
                    *[{"type": "image_url", "image_url": {"url": url}} for url in image_urls],
                    {"type": "text", "text": EXTRACT_PROMPT},
                ],
            },
        ],
        "max_tokens": 16384,
        "temperature": 0.0,
        "frequency_penalty": 0.3,
    }).encode("utf-8")

    response = target.raw_predict(
        body=body,
        headers={"Content-Type": "application/json"},
        use_dedicated_endpoint=True,
    )
    assert response.status_code == 200, response.text
    return response.json()["choices"][0]["message"]["content"]


def extract_pdf(target: aiplatform.Endpoint) -> str:
    """Load the PDF and extract text from every page in batches via the VLM."""
    pages = load_pdf_pages(PDF_PATH)
    all_urls = [encode_pil_to_data_url(p) for p in pages]
    extracted = []
    for i in range(0, len(all_urls), BATCH_SIZE):
        batch = all_urls[i : i + BATCH_SIZE]
        print(f"  VLM processing pages {i + 1}-{i + len(batch)} of {len(all_urls)}...")
        extracted.append(vlm_extract(target, batch))
    return "\n\n".join(extracted)

## 5. Parsing Gemma's `tool_code` Blocks

Gemma's native function-calling format wraps tool invocations in fenced `tool_code` blocks containing plain Python, e.g.:

```tool_code
parse_pdf_tool(filename="report.pdf")
```

We use Python's `ast` module to safely extract the function name and keyword arguments. Gemma sometimes wraps the call in `print()` — we handle that too.

In [ ]:
TOOL_CODE_RE = re.compile(r"```tool_code\s*(.*?)\s*```", re.DOTALL)


def parse_tool_code(code: str) -> tuple[str, dict[str, Any]]:
    """
    Parse a Python function-call string into (function_name, kwargs).
    Unwraps print() wrappers if present.
    """
    tree = ast.parse(code.strip(), mode="eval")
    call = tree.body
    if not isinstance(call, ast.Call):
        raise ValueError(f"Expected a function call, got: {ast.dump(call)}")

    # Unwrap print() — Gemma often generates print(tool(...))
    if isinstance(call.func, ast.Name) and call.func.id == "print":
        if call.args and isinstance(call.args[0], ast.Call):
            call = call.args[0]
        else:
            raise ValueError(f"print() does not wrap a function call: {code}")

    if isinstance(call.func, ast.Name):
        func_name = call.func.id
    elif isinstance(call.func, ast.Attribute):
        func_name = call.func.attr
    else:
        raise ValueError(f"Unsupported call target: {ast.dump(call.func)}")

    args: dict[str, Any] = {}
    for kw in call.keywords:
        args[kw.arg] = ast.literal_eval(kw.value)
    return func_name, args

## 6. `ToolCodeGemma` — the ADK Model Adapter

The ADK expects models to return structured `FunctionCall` objects, but our Gemma endpoint produces free-form text with `tool_code` blocks. This adapter bridges the two:

| Direction | What happens |
|---|---|
| **ADK to Gemma** | Describes registered tools in the system prompt (cookbook style). Formats prior `FunctionResponse` as `tool_output` blocks. |
| **Gemma to ADK** | Parses `tool_code` blocks from the response text into `types.FunctionCall` parts the Runner can dispatch. |

No OpenAI `tools` parameter is used — this is pure prompt-based function calling.

In [7]:
TOOL_CODE_SYSTEM = (
    "At each turn, if you decide to invoke any of the function(s), "
    "it should be wrapped with ```tool_code```. "
    "The python methods described below are imported and available, "
    "you can only use defined methods. "
    "The generated code should be readable and efficient. "
    "When a function has arguments, always pass every required argument. "
    "If the user mentions a PDF filename, pass that exact value to the filename argument. "
    "The response to a method will be wrapped in ```tool_output``` "
    "use it to call more tools or generate a helpful, friendly response."
)


def _build_tool_prompt(tool_declarations: list[types.FunctionDeclaration]) -> str:
    """Convert ADK function declarations into the cookbook system-prompt format."""
    type_map = {"string": "str", "integer": "int", "number": "float", "boolean": "bool"}
    blocks: list[str] = []
    for decl in tool_declarations:
        params_dict = (
            decl.parameters.model_dump(exclude_none=True)
            if decl.parameters
            else {"type": "object", "properties": {}}
        )
        props = params_dict.get("properties", {})
        required = set(params_dict.get("required", props.keys()))

        sig_parts = [f"{p}: {type_map.get(s.get('type', 'string'), 'str')}" for p, s in props.items()]
        sig = f"def {decl.name}({', '.join(sig_parts)}) -> str:"
        doc = decl.description or ""

        param_lines = [
            f"        {p} ({type_map.get(s.get('type','string'),'str')}, {'required' if p in required else 'optional'}): {s.get('description','')}"
            for p, s in props.items()
        ]
        param_section = ""
        if param_lines:
            param_section = "\n\n    Args:\n" + "\n".join(param_lines) + "\n"

        blocks.append(f"```python\n{sig}\n    \"\"\"\n    {doc}{param_section}    \"\"\"\n```")
    return "\n".join(blocks)


class ToolCodeGemma(BaseLlm):
    """ADK model adapter using Gemma's native tool_code blocks."""

    endpoint_name: str

    async def generate_content_async(
        self, llm_request: LlmRequest, stream: bool = False
    ) -> AsyncGenerator[LlmResponse, None]:
        endpoint = get_endpoint(self.endpoint_name)
        client = get_gemma_client(endpoint)

        # Collect function declarations for the system prompt
        all_decls: list[types.FunctionDeclaration] = []
        if llm_request.config and llm_request.config.tools:
            for tool in llm_request.config.tools:
                if isinstance(tool, types.Tool) and tool.function_declarations:
                    all_decls.extend(tool.function_declarations)

        # Build system prefix
        sys_parts: list[str] = []
        if llm_request.config and llm_request.config.system_instruction:
            sys_parts.append(llm_request.config.system_instruction)
        if all_decls:
            sys_parts.append(TOOL_CODE_SYSTEM)
            sys_parts.append("The following Python methods are available:\n" + _build_tool_prompt(all_decls))
        system_prefix = "\n\n".join(sys_parts).strip()

        # Convert ADK contents to OpenAI messages
        messages: list[dict[str, str]] = []
        for content in llm_request.contents:
            segments: list[str] = []
            for part in content.parts:
                if part.text:
                    segments.append(part.text)
                if part.function_call:
                    arg_strs = ", ".join(
                        f'{k}="{v}"' if isinstance(v, str) else f"{k}={v}"
                        for k, v in part.function_call.args.items()
                    )
                    segments.append(f"```tool_code\n{part.function_call.name}({arg_strs})\n```")
                if part.function_response:
                    resp = part.function_response.response
                    resp_text = json.dumps(resp, ensure_ascii=False) if isinstance(resp, dict) else str(resp)
                    segments.append(f"```tool_output\n{resp_text}\n```")

            if not segments:
                continue
            role = "assistant" if content.role == "model" else "user"
            text = "\n".join(segments)

            if role == "user" and system_prefix:
                text = f"{system_prefix}\n\n{text}"
                system_prefix = ""

            if messages and messages[-1]["role"] == role:
                messages[-1]["content"] += f"\n\n{text}"
            else:
                messages.append({"role": role, "content": text})

        if system_prefix:
            if messages and messages[0]["role"] == "user":
                messages[0]["content"] = f"{system_prefix}\n\n{messages[0]['content']}"
            else:
                messages.insert(0, {"role": "user", "content": system_prefix})

        # Call Gemma (plain chat, no tools param)
        response = await asyncio.to_thread(
            lambda: client.chat.completions.create(
                model=GEMMA_MODEL_ID,
                messages=messages,
                temperature=0.0,
                max_tokens=4096,
            )
        )

        text = response.choices[0].message.content or ""
        print(f"\n--- Gemma response ---\n{text}\n")

        # Parse tool_code if present
        match = TOOL_CODE_RE.search(text)
        if match:
            func_name, func_args = parse_tool_code(match.group(1))
            yield LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part(function_call=types.FunctionCall(name=func_name, args=func_args))],
                )
            )
            return

        # Plain text response
        yield LlmResponse(
            content=types.Content(role="model", parts=[types.Part.from_text(text=text)])
        )

## 7. Define the Tool

This is the Python function that the ADK Runner will call when Gemma decides to invoke `parse_pdf_tool`. It renders the PDF to images and sends them to the Nemotron VLM for text extraction.

In [8]:
def parse_pdf_tool(filename: str) -> str:
    """
    Tool that parses a PDF file using the Nemotron VLM.

    Args:
        filename (str): The name or path of the PDF file to parse.
    """
    print(f"  [parse_pdf_tool] Extracting '{filename}' with Nemotron VLM...")
    vlm_endpoint = get_endpoint(VLM_ENDPOINT_NAME)
    return extract_pdf(vlm_endpoint)

## 8. Initialize the ADK Agent and Runner

Wire everything together:
- **`ToolCodeGemma`** is the model that talks to the Gemma endpoint.
- **`Agent`** binds the model to the tool (`parse_pdf_tool`).
- **`Runner`** manages the multi-turn loop: user message -> model -> tool dispatch -> model -> final answer.
- **`Productionizing Memory`** here we use ADK's InMemorySessionService for the demo. In a production environment, ADK allows you to hot-swap this with persistent storage (like Firestore, SQLite, or Cloud SQL) to maintain user conversation history across sessions with zero changes to the agent logic.

In [ ]:
model = ToolCodeGemma(
    model=GEMMA_MODEL_ID,
    endpoint_name=GEMMA_ENDPOINT_NAME,
)

agent = Agent(
    name="pdf_agent",
    model=model,
    description="Agent that parses PDFs via VLM",
    instruction=(
        "You are a strict financial auditor. Answer strictly based on the provided text. "
        
        "1. SCOPE: "
        "   - Answer only the specific metrics requested. "
        
        "2. SOURCE OF TRUTH (Hierarchy): "
        "   - High-Level Metrics (Revenue, Net Income): Use 'GAAP/Non-GAAP P&L' tables. "
        "   - Specific Segments/Breakdowns: If asked for a specific segment (like 'Compute' "
        "     or 'Capital Return'), look for detailed CHARTS or breakdown slides. "
        "     Do not just quote the total Company Revenue. "
        
        "3. DUAL REPORTING: "
        "   - Always report BOTH GAAP and Non-GAAP for P&L items (Net Income, EPS, Margins). "
        "   - Revenue is typically the same. "
        
        "4. YEAR ALIGNMENT: "
        "   - Verify column headers match the requested year (e.g., Q3 FY26)."
        
        "5. METRIC DEFINITIONS: "
        "   - 'Free Cash Flow' is Non-GAAP. "

        "6. UNIT PRECISION: "
        "   - Billions ($B) vs Per Share ($). "
      
    ),
    tools=[parse_pdf_tool],
)

session_service = InMemorySessionService()
runner = Runner(
    app_name="pdf-parsing-agent",
    agent=agent,
    session_service=session_service,
)

print("Agent and Runner initialized.")

## **Architecture Visualizer**

In [ ]:
import uuid
from IPython.display import display, HTML

def visualize_adk_event_loop(adk_agent):
    """Generates a detailed Sequence Diagram showing the true ADK Runner event loop."""
    
    graph_def = [
        "sequenceDiagram",
        "    autonumber",
        "    actor User",
        "    participant Runner as ADK Runner<br/>(Event Loop & State)",
        f"    participant Gemma as {adk_agent.name}<br/>(Gemma 3 Orchestrator)",
        "    participant Tool as parse_pdf_tool<br/>(Nemotron VLM)",
        "    ",
        "    User->>+Runner: runner.run_async(user_message)",
        "    Note right of Runner: Loads Session &<br/>InvocationContext",
        "    Runner->>+Gemma: Evaluates Query + Context",
        "    Gemma-->>-Runner: Yields Event (FunctionCall)",
        "    Runner->>+Tool: Executes Python tool logic",
        "    Note right of Tool: Renders PDF to images<br/>Extracts & aligns tables",
        "    Tool-->>-Runner: Returns FunctionResponse (Text)",
        "    Note right of Runner: Updates Session History",
        "    Runner->>+Gemma: Sends updated Context",
        "    Gemma-->>-Runner: Yields Event (Final Text)",
        "    Runner-->>-User: Streams Final Answer"
    ]

    mermaid_code = "\n".join(graph_def)
    div_id = f"mermaid_{uuid.uuid4().hex}"
    
    html_template = f"""
    <div class="mermaid" id="{div_id}" style="text-align: center;">
        {mermaid_code}
    </div>
    <script type="module">
        import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
        mermaid.initialize({{ startOnLoad: true, theme: 'base' }});
        mermaid.init(undefined, document.getElementById('{div_id}'));
    </script>
    """
    display(HTML(html_template))

print("👇 ADK Multi-Turn Execution Flow (Event Loop):")
visualize_adk_event_loop(agent)

## 9. Run the Agent

Send a question to the agent. The ADK Runner handles the full loop automatically:

1. Gemma reads the question and generates a `tool_code` block calling `parse_pdf_tool`.
2. The Runner dispatches the tool — Nemotron VLM processes all PDF pages.
3. The extracted text is sent back to Gemma as `tool_output`.
4. Gemma produces a final answer grounded in the document.

In [ ]:
from PIL import Image
async def ask_agent(question: str) -> str:
    """Send a question to the agent and return the final text answer."""
    session = await session_service.create_session(
        app_name="pdf-parsing-agent", user_id="user"
    )
    user_msg = types.Content(
        role="user",
        parts=[types.Part.from_text(text=question)],
    )

    print(f"Question: {question}\n")

    responses: list[str] = []
    async for event in runner.run_async(
        user_id="user",
        session_id=session.id,
        new_message=user_msg,
    ):
        if not (event.content and event.content.parts):
            continue
            
        for part in event.content.parts:
            # Highlight ADK Tool Dispatch
            if part.function_call:
                print(f" [ADK Router] Dispatching Tool: {part.function_call.name}()")
            
            # Highlight ADK Tool Return
            if part.function_response:
                print(f" [ADK Router] Tool Output Received. Returning context to Gemma...")
            
            # Capture final text
            if part.text:
                responses.append(part.text)

    return " ".join(responses)


answer = await ask_agent(
    "Please parse v2-NVDA-F3Q26-Quarterly-Presentation.pdf and tell me "
    "what were NVIDIA's Revenue ($) and Net Income ($) in Q3 FY26 were, "
    "and how did they compare to Revenue ($) and Net Income ($) in Q3 FY25?"
)

print(f"\n{'=' * 70}")
print(f"Final Answer:\n\n{answer}")
print(f"{'=' * 70}")